In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import nltk
import spacy as sp
from bs4 import BeautifulSoup
import re

import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

In [2]:
sp.prefer_gpu()

True

In [ ]:
path = "/content/16k_Movies.csv"

data = pd.read_csv(path).drop("Unnamed: 0", axis=1)
data.dropna(inplace=True)
data.drop_duplicates(inplace=True)
data.head()

In [4]:
data["No of Persons Voted"] = data["No of Persons Voted"].str.replace(",", "")
data["No of Persons Voted"] = data["No of Persons Voted"].astype(int)
data['Release Date'] = pd.to_datetime(data['Release Date'], errors='coerce')
data = data.assign(Genres=data.Genres.str.split(',')).explode('Genres')
data['Genres'] = data['Genres'].str.strip()

In [5]:
print("Mean", data["No of Persons Voted"].mean())
print("Median", data["No of Persons Voted"].median())
print("Max", data["No of Persons Voted"].max())
print("Min", data["No of Persons Voted"].min())

Mean 161.31586590826575
Median 43.0
Max 10639
Min 4


In [ ]:
df = data[data["No of Persons Voted"] >= 100]
director_counts = df["Directed by"].value_counts()
df = df[df["Directed by"].isin(director_counts[director_counts >= 4].index)]
df.head()

In [7]:
data_group = df.groupby("Directed by")["Rating"].mean()
data_group = data_group.sort_values(ascending=False)[:30]
data_group

,Rating
Directed by,
Hal Ashby,9.700000
Hadi Hajaig,9.600000
Andrea Di Stefano,9.500000
"Terry L. Noss, \n \n Richard Rich",9.500000
Swati Bhise,9.500000
D.M.W. Greer,9.500000
Andrew Morahan,9.400000
Kenneth Johnson,9.400000
Jim Sonzero,9.300000


In [8]:
def movies_directed_by(name):
  movies = df[df["Directed by"] == name]
  movies = movies.sort_values(by="Rating", ascending=False)
  return movies.drop("Directed by",axis=1)

In [ ]:
movies_directed_by("Hayao Miyazaki")

In [10]:
def get_top_movies_by_genre(genre, dataframe=df, top=5):
    # Filter movies by genre
    movies = dataframe[dataframe["Genres"].str.contains(
        genre, case=False, na=False)]

    # Sort movies by 'No of Persons Voted'
    movies = movies.sort_values(by="Rating", ascending=False)

    # Remove duplicates based on 'Title'
    movies.drop_duplicates(subset=['Title'], inplace=True)

    # Select top N movies and drop the 'Genres' column
    movies = movies.head(top).drop("Genres", axis=1)

    # Reset the index and return the dataframe
    return movies.reset_index(drop=True)

In [ ]:
get_top_movies_by_genre(genre="Horror", top=10)

In [ ]:
genre_ratings = df.groupby('Genres')['Rating'].mean()

plt.figure(figsize=(12, 6))
plt.bar(genre_ratings.index, genre_ratings.values)
plt.xlabel('Genres')
plt.ylabel('Average Rating')
plt.title('Average Rating by Genre')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [13]:
nlp = sp.load("en_core_web_sm")

/usr/local/lib/python3.10/dist-packages/spacy/util.py:1740: UserWarning: [W111] Jupyter notebook detected: if using `prefer_gpu()` or `require_gpu()`, include it in the same cell right before `spacy.load()` to ensure that the model is loaded on the correct device. More information: http://spacy.io/usage/v3#jupyter-notebook-gpu
  warnings.warn(Warnings.W111)


In [14]:
def clean_text(text):
    # Remove multiple spaces and leading/trailing spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Convert to lowercase
    text = text.lower()

    # Tokenize with spaCy and remove stop words and punctuation
    doc = nlp(text)
    words = [token.text for token in doc if not token.is_stop and not token.is_punct]

    # Join the remaining words into a string
    description = " ".join(words)

    return description

In [15]:
example = data["Description"][0]
print(example)
print()
print(clean_text(example))

This masterwork by Krzysztof Kieślowski is one of the twentieth century’s greatest achievements in visual storytelling. Originally made for Polish television, Dekalog focuses on the residents of a housing complex in late-Communist Poland, whose lives become subtly intertwined as they face emotional dilemmas that are at once deeply personal and universally human. Its ten hour-long films, drawing from the Ten Commandments for thematic inspiration and an overarching structure, grapple deftly with complex moral and existential questions concerning life, death, love, hate, truth, and the passage of time. Shot by nine different cinematographers, with stirring music by Zbigniew Preisner and compelling performances from established and unknown actors alike, Dekalog arrestingly explores the unknowable forces that shape our lives. Also available are the longer theatrical versions of the series’ fifth and sixth films: A Short Film About Killing and A Short Film About Love. [Janus Films]

masterwo

In [16]:
descriptions = df["Description"].apply(clean_text)
ratings = df["Rating"].values
ratings = ratings.reshape(-1, 1)

In [17]:
max_len = max(len(description.split()) for description in descriptions)
print(f'Maximum length of a description: {max_len}')

vectorizer = TextVectorization(output_mode='int', split='whitespace',
                               standardize='strip_punctuation', max_tokens=10000,
                               output_sequence_length=max_len)

vectorizer.adapt(descriptions)
print(f'Vocabulary size: {vectorizer.vocabulary_size()}')

Maximum length of a description: 144
Vocabulary size: 10000


In [18]:
def generate_dataset(descriptions_vectorized, score, buffer_size, batch_size, p):
    assert len(descriptions_vectorized) == len(score), "Features and labels must have the same length."
    assert 0 <= p <= 1, "p must be between 0 and 1."
    # Calculates the split size between training and testing
    size = int(p * len(descriptions_vectorized))

    # Creates the dataset from features and scores
    dataset = tf.data.Dataset.from_tensor_slices((descriptions_vectorized, score))

    dataset = dataset.shuffle(buffer_size=buffer_size)

    # Split the dataset into training and testing
    dataset_train = dataset.take(size)
    dataset_test = dataset.skip(size)

    # Apply batch and prefetch after split
    dataset_train = dataset_train.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    dataset_test = dataset_test.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return dataset_train, dataset_test

In [19]:
descriptions_vectorized=vectorizer(descriptions)
buffer_size=10000
batch_size=32
p=0.8


train_df, test_df = generate_dataset(descriptions_vectorized=descriptions_vectorized,
                                      score=ratings,
                                      buffer_size=buffer_size,
                                      batch_size=batch_size,
                                      p=p)

In [20]:
def create_model(vocab_size, embedding_dim, rnn_units, lr=0.001):
  model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, embedding_dim),
    tf.keras.layers.LSTM(rnn_units, return_sequences=True),
    tf.keras.layers.Dense(1)
  ])
  optim = tf.keras.optimizers.Adam(learning_rate=lr)
  loss = tf.keras.losses.Huber()

  model.compile(loss=loss,
                optimizer=optim,
                metrics=['mae', 'mse'])
  return model

In [21]:
vocab_size = vectorizer.vocabulary_size()
embedding_dim = 128
rnn_units = 128
model = create_model(vocab_size, embedding_dim, rnn_units)

In [22]:
patience = 5
verbose = 1
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=patience, verbose=verbose)

In [23]:
history = model.fit(train_df,
                    epochs=10,
                    verbose=verbose,
                    validation_data=test_df,
                    callbacks=[early_stopping])

Epoch 1/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - loss: 1.4280 - mae: 1.8460 - mse: 7.6492 - val_loss: 0.7131 - val_mae: 1.1186 - val_mse: 2.3881
Epoch 2/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.7143 - mae: 1.1211 - mse: 2.3053 - val_loss: 0.7083 - val_mae: 1.1112 - val_mse: 2.2886
Epoch 3/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 0.6791 - mae: 1.0814 - mse: 2.1006 - val_loss: 0.4232 - val_mae: 0.7853 - val_mse: 1.1580
Epoch 4/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.3736 - mae: 0.7028 - mse: 1.1107 - val_loss: 0.1678 - val_mae: 0.4102 - val_mse: 0.4580
Epoch 5/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - loss: 0.1661 - mae: 0.4031 - mse: 0.4621 - val_loss: 0.0838 - val_mae: 0.2551 - val_mse: 0.2151
Epoch 6/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.0889 - mae: 0.2601 - mse: 0.2399 - val_loss: 0.0619 - val_mae: 0.2111 - val_mse: 0.1623
Epoch 7/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 0.0556 - mae: 0.1973 - mse: 0.141

In [24]:
def plot_history(history, metric):
    plt.plot(history.history[metric])
    plt.plot(history.history['val_' + metric])
    plt.xlabel("Epochs")
    plt.ylabel(metric)
    plt.legend([metric, 'val_' + metric])
    plt.show()

In [ ]:
plot_history(history, 'loss')
plot_history(history, 'mae')
plot_history(history, 'mse')